# Applications of Machine Learning Methods to MD Data Analysis

This workbook showcases the application of two ML methods - **Principal Component Analysis (PCA)**, and **Clustering** - to the analysis of MD data.

**PCA** provides an approach to a graphically-intuitive representation of the snapshots in a trajectory as points in a low-dimensional (here just 2D) space.

**Clustering** provides methods to separate the snapshots in a trajectory into a (probably) small number of representative classes, where all the snapshots in each class are structurally rather similar to each other.

Although both ML methods can be applied to MD data independently, you will see that often they work rather well together.


--------
## 1. Principal Component Analysis

We begin by importing the Python packages required: `matplotlib` for the graphics, `mdtraj` to process the MD data, `numpy` for numerical methods and data manipulation, `nglview` for molecular graphics and `[MDPlus](https://bitbucket.org/claughton/mdplus/src/master/) for PCA:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import mdtraj as mdt
from mdplus.pca import PCA
import nglview as nv

We will use the same Chignolin MD simulation data as in the previous "Analysis of Protein Simulations" workshop.

To recap, this is 20,000 snapshots from a 20 nanosecond simulation of this short (10 residue) peptide that has been shown experimentally to "flip" between a range of conformational states:

In [ ]:
t = mdt.load('chignolin.xtc', top='chignolin.pdb')
print(t)

As a reminder, let's produce the same RMSD plot as last time, which reveals (if somewhat inprecisely) these conformational transitions and alternative states:

In [ ]:
rmsd = mdt.rmsd(t, t[0])
plt.plot(t.time, rmsd)
plt.xlabel('time (ps)')
plt.ylabel('RMSD (nm)')

Now we do the PCA. 

 - The first line below creates an instance of a PCA transformer. 

 - The second line does two things - it uses the trajectory coordinate data (`t.xyz`) to "fit" the transformer (calculating the set of eigenvectors that together could rotate the atoms' Cartesian coordinates into the optimal PC set), and then actually applies this transformation to each snapshot in the trajectory, producing an [*n_frames*, *3\*n_atoms*] matrix of scores.

 - The third line onwards produces a plot of the value of the first principal component in each frames of the trajectory, against time.


Compare with the RMSD plot above!

In [ ]:
p = PCA()
scores = p.fit_transform(t.xyz)
plt.plot(t.time, scores[:, 0])
plt.xlabel('time (ps)')
plt.ylabel('PC0 (nm)')

What does the time series for the second principal component look like?

In [ ]:
plt.plot(t.time, scores[:, 1])
plt.xlabel('time (ps)')
plt.ylabel('PC1 (nm)')

Interesting - while PC0 captures something quite similar to what RMSD was capturing, PC1 is capturing something quite different.

How many other PCs might be capturing "interesting" dynamics data?

Below we plot the values of the first 25 eigenvalues from ther PCA analysis - that is, the variance in the trajectory data that each of the first 25 PCs (or eigenvectors) explains:

In [ ]:
plt.plot(p.eigenvalues[:25])
plt.xlabel('PC #')
plt.ylabel('eigenvalue')

It's clear that only a very small number of the top PCs contribute significantly to the variance in the trajectory data (i.e., the dynamics of the system). 

To check this, let's plot the time series for the 20th PC:

In [ ]:
plt.plot(t.time, scores[:, 19])
plt.xlabel('time (ps)')
plt.ylabel('PC20 (nm)')

This doesn't look like much more than low-amplitude random noise.

On the basis of this, we might conclude that even describing each snapshot as a point in the 2D PC0/PC1 plane might not be a terrible representation of the data - and this can be plotted!

In [ ]:
plt.plot(scores[:, 0], scores[:, 1], '.')
plt.xlabel('PC0')
plt.ylabel('PC1')

There is clearly a lot of information in this 2D representation of the conformational space of chignolin. There seems to be clustering of the conformations into a limited number of highly populated regions. We might wonder if this is the result of the form of the underlying free energy surface.

We can produce a visualization of this by using `numpy` methods to divide the PC0/PC1 surface into rectangular bins, calculate the number of observations in each bin, then take the negative of the log of this as an energy scale (units of kT):

In [ ]:
H, xedges, yedges = np.histogram2d(scores[:, 0], scores[:, 1], bins=50)
plt.imshow(-np.log(H.T), origin='lower', extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
plt.xlabel('PC0')
plt.ylabel('PC1')
interesting_points = [[-3, 0], [1.5, -2.5], [2, 2]]
labels = ['A', 'B', 'C']
for i in range(3):
    plt.text(*interesting_points[i], labels[i], size=12, color='white')

We can see at least three major low-energy regions (A, B, C), plus maybe a few extra minor ones. What sorts of chignolin conformations correspond to each of these?

The cell below defines a little function to find the index of the snapshot that has it's first two PCs closest to each labelled point, then creates a small trajectory with just these snapshots, then visualizes it. The slider bar can be used to switch between representatves of basin A, B, and C:

In [ ]:
def pick_conf(scores, xy):
    dx = scores[:, :2] - xy
    dr = np.sqrt((dx*dx).sum(axis=1))
    return np.argmin(dr)

interesting_indices = []
for interesting_point in interesting_points:
    interesting_indices.append(pick_conf(scores, interesting_point))
    
v_i = nv.show_mdtraj(t[interesting_indices])
v_i.add_representation('licorice', 'protein')
v_i

They are clearly quite different from each other.

In summary, PCA has revealed a low-dimensional free energy surface for chignolin that features a few quite well-defined local minima. By eye we have pulled out representative structures for three of these.

--------------
## 2. Clustering

Now we investigate the application of more rigorous ML-based clustering methods to the analysis of this data.

### Background

Although algorithms for clustering can be rigorously defined, the definition of a 'correct' clustering process is nearly always impossible. Clustering is therefore typically an iterative and interactive process in whcih a range of methods, with a range of parameters, is investigated in order to arrive at a final approach that seems the best for the job required.

Most clustering methods can be defined as *unsupervised*, in that the number of clusters the data should be separated into is not defined by the user in advance. However, one of the most popular methods - **K-means** - is the exception to this, and so can be regarded as *supervised*. Before we look at **K-means**, we will look at the *unsupervised* **Ward** approach, which uses *agglomerative clustering*

#### Aggolmerative clustering
In agglomerative clustering, we begin with a **distance matrix** which defines how far each obervation (here, conformation of the molecule) is from every other one, a **linkage algorithm** that decides if a number of observations are part of the same cluster or not based on the distances between them, and a **threshold** distance which is set to zero. Because the thrshold distance is zero, the algorithm does not regard any two observations as belonging to the same cluster, so if there are N observations, there are *N* clusters. Now the threshold distance is raised until it reaches a value where the algorithm says that the closest two observations now belong to one cluster - so now there are *N-1* clusters. Then the threshold is raised again, until two other observations merge into a cluster, or a third observation joins the first cluster, and now there are *N-2* clusters. The process is continued, stepping up the threshold value so at each iteration the number of clusters decrases by one, until everything eventually merges into one cluster. With this process complete, it is then up to the user to decide at which agglomeration step, associated with a particular value of the threshold distance, to stop and so how many clusters should be defined. There are a variety of linkage algorithms that can be used to decide if two observations/clusters should merge or not, e.g. **single**, **complete**, and **average**, but **Ward** is a popular choice (see [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html#scipy.cluster.hierarchy.linkage) for more details).

### Ward agglomeratve clustering

We begin by importing the required methods from `scipy`:

In [ ]:
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import ward, fcluster

The first job is to create the distance matrix. This will contain the RMSD between every pair of conformations. As there are 20,000 snapshots this is 400 million RMSD values to calculate. Actually we only need to calculate less than half of these because `rmsd[i, j] == rmsd[j, i]` and `rmsd[i, i] = 0`. But you shoud still be prepared for a slow calculation:

In [ ]:
rmsdlist = []
percent = 0
for i in range(t.n_frames - 1):
    rmsdlist.append(mdt.rmsd(t[i+1:], t[i]))
    if i % (t.n_frames // 10) == 0:
        print(f'{percent}% done...')
        percent += 10
        
rmsds = np.concatenate(rmsdlist)

Now we can do the Ward clustering. The results are returned as a *linkage matrix*:

In [ ]:
zward = ward(rmsds)

The next cell takes data from the linkage matrix and produces a plot that shows at what values of the threshold distance each agglomeration event happens. Visual inspection of this can help identify (with luck) a "natural" value that separates clearly distinct clusters from a something closer to a continuum:

In [ ]:
plt.plot(zward[:, 2], 'o')
plt.plot([0.0, 20000], [12.0, 12.0], '--') # this turns out to look reasonable
plt.xlabel('partition number')
plt.ylabel('distance threshold')

The orange line at distance threshold 12.0 marks one possible cut-off, but later you should invesigate others.

Next we use this distance threshold with the linkage matrix to assign each observation in the dataset to one of the clusters. The output is a list of "labels", so for example if `labels[23]` was `4`, that would mean that observation 23 belongs to cluster `4`.

We count up the number of observations in each cluster:

In [ ]:
ward_labels = fcluster(zward, 12.0, criterion='distance')
for i in set(ward_labels):
    print(f'cluster {i} contains {(ward_labels == i).sum()} snapshots')

now we can use the cluster labels to colour-code the points on the PCA-dervived maps we generated earlier:

In [ ]:
ward_clusters = [scores[ward_labels == i] for i in set(ward_labels) if i != 0]
for i, w in enumerate(ward_clusters):
    l = len(w)
    plt.plot(w[:, 0], w[:, 1], '.', label=f'cluster {i+1} (n={l})')
plt.legend()
plt.xlabel('PC0')
plt.ylabel('PC1')

Do the cluster assignments look reasonable to you?

Let's identify the index of the snapshot in each cluster that is closest to its centre (the one with the smallest average distance to all other snapshots in the same cluster):

In [ ]:
cluster_centres = []
for i in set(ward_labels):
    mask = np.where(ward_labels == i)[0]
    r1 = squareform(rmsds)[mask][:, mask]
    rm = r1.mean(axis=0)
    irm = np.argmin(rm)
    cluster_centres.append(int(mask[irm]))
print(cluster_centres)

Now lets add labels to the plot to show where they are:

In [ ]:
centre_labels = 'ABCDEFHIJK'
for i, w in enumerate(ward_clusters):
    l = len(w)
    c = cluster_centres[i]
    plt.plot(w[:, 0], w[:, 1], '.', label=f'cluster {i+1} (n={l})')
    plt.text(scores[c, 0], scores[c, 1], centre_labels[i], color='white', size=12)
plt.legend()
plt.xlabel('PC0')
plt.ylabel('PC1')

And generate images for each of them, as before. How similar are they to the structures you pulled out "by eye" from just looking at the PCA map?

In [ ]:
v_i = nv.show_mdtraj(t[cluster_centres])
v_i.add_representation('licorice', 'protein')
v_i

----------------
### K-means clustering

Now let's compare with the results we get when we use a K-means clustering approach instead.

We import the required method from scipy:

In [ ]:
from scipy.cluster.vq import kmeans2

K-means clustering doesn't use a pre-computed distance matrix, instead we have to superimpose all the snapshots in our trajectory (to remove any global rotation or translation from the dynamics), and then reshape the coordinate data into a 2-D matrix of 'observation vectors':

In [ ]:
tf = t.superpose(t[0])
observations = tf.xyz.reshape((t.n_frames, -1))

Now we can do the clustering, but we have to choose a number of clusters to generate. This could be difficult if we hadn't already done the PCA-based analysis at least.

The method returns the centroids - the coordinates of the geometrical centre of each cluster (which will probably not be exactly the same as the coordinates of any real shapshot), and a list of labels, similar to what `ward` did:

In [ ]:
n_clusters = 4 # You might want to experiment with changing this, later
centroids, klabels = kmeans2(observations, n_clusters)
for i in set(klabels):
    print(f'cluster {i} contains {(klabels == i).sum()} snapshots')

In [ ]:
k_clusters = [scores[klabels == i] for i in set(klabels)]

for i, w in enumerate(k_clusters):
    l = len(w)
    plt.plot(w[:, 0], w[:, 1], '.', label=f'cluster {i} (n={l})')
plt.legend()
plt.xlabel('PC0')
plt.ylabel('PC1')

How do the results compare with those from Ward clustering? Be aware that the colour-coding of the clusters may be different...

-----------

## 3. Over to you.

All the analysis done here has used the difference between **all** atoms in each snapshot as a measure of the distance between them. But for clustering, we might only be interested in structures that have a different backbone conformation, irrespective of where side chains are.

Go back to the beginning of the notebook, and use the MDTraj methods you know to strip the trajectory down to just backbone atoms. Then explore the effect of this on both the PCA-based graphical analysis and the clustering.

--------------
## Summary

**Machine learning** methods can be powerful for MD data analysis. **PCA** provides a route to a graphically intuitive representation of how simulations have explored conformational space, and can provide quantitative metrics too.

**Clustering** is a very useful tool for simplifying complex data and identifying small numbers of key structures whose detailed examination can summarise key structural features that differentiate between different states of the system.

**K-means clustering** can be considerably faster than **agglomerative clustering** methods such as **Ward** for large data sets, because no RMSD distance matrix needs to be precomputed. However it requires the user to choose the number of clusters to generate, without necessarily having any idea what the "right" number might be.